# Part 02 — Capacity and C-rate

Capacity tells you **how much electric charge a cell is rated to deliver under specified test conditions**. C-rate tells you how large the applied current is relative to that rated capacity.

By the end of this notebook, you should be able to answer three questions:

- What current is 2C for a 20 Ah cell?
- What is the ideal duration associated with 2C?
- Why can a real cell reach its voltage limit before or after that ideal duration?

The illustrative cell used throughout this notebook has a nominal capacity of **20 Ah**.

This lesson uses the tested `battery_core` package as the source of truth for all calculations.

## 1. A C-rate connects current, capacity, and time

For current magnitude $I$ in amperes and nominal capacity $Q_\mathrm{nominal}$ in ampere-hours,

$$
C_\mathrm{rate}=\frac{I}{Q_\mathrm{nominal}}
$$

When current is in A and capacity is in Ah, the numerical C-rate has units of reciprocal hours:

$$
\frac{[\mathrm{A}]}{[\mathrm{A}\cdot\mathrm{h}]}
=\mathrm{h}^{-1}
$$

Battery engineers commonly write values such as **C/10**, **1C**, **2C**, and **10C**. The corresponding ideal relationships are

$$
I=C_\mathrm{rate}Q_\mathrm{nominal},
\qquad
t_\mathrm{ideal}=\frac{1}{C_\mathrm{rate}}.
$$

The second equation does not contain capacity because C-rate already normalizes the current by capacity. At the same C-rate, cells of different capacities have the same ideal duration but different currents.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from battery_core import (
    c_rate_from_current,
    current_from_c_rate,
    ideal_duration_hours,
)

NOMINAL_CAPACITY_AH = 20.0
C_RATES = np.array([0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0])
RATE_LABELS = ["C/10", "C/5", "C/2", "1C", "2C", "5C", "10C"]


def pretty_time(hours: float) -> str:
    """Format a duration in hours or minutes for display."""
    return f"{hours:.2f} h" if hours >= 1.0 else f"{hours * 60.0:.0f} min"


currents_a = current_from_c_rate(NOMINAL_CAPACITY_AH, C_RATES)
durations_h = ideal_duration_hours(C_RATES)

print(f"Nominal capacity: {NOMINAL_CAPACITY_AH:.0f} Ah\n")
print(f"{'Rate':>7}  {'Current [A]':>11}  {'Ideal duration':>14}")
print("-" * 38)
for label, current, duration in zip(RATE_LABELS, currents_a, durations_h):
    print(f"{label:>7}  {current:11.1f}  {pretty_time(float(duration)):>14}")

## 2. Reading the table

Three patterns are important:

1. **1C is the reference point.** For this 20 Ah cell, 1C corresponds to 20 A and an ideal duration of 1 hour.
2. **Doubling C-rate doubles current and halves ideal duration.**
3. **Ideal duration depends on C-rate, not on the absolute cell capacity.** A 3 Ah cell and a 300 Ah cell both have an ideal duration of 30 minutes at 2C, but their currents are 6 A and 600 A, respectively.

That normalization is why C-rate is useful when comparing cells of very different sizes.

## 3. The same table shown as two linked graphs

The graphs below use exactly the C-rates from the table. Each marker is one table row:

- the left panel shows how current increases with C-rate for the 20 Ah cell;
- the right panel shows how ideal duration decreases with C-rate.

Both axes are logarithmic so that C/10 through 10C remain visible on the same figure. The point labels give the actual table values, so the graph can be read without estimating from the axes.

In [ ]:
plot_rates = np.logspace(-1, 1, 300)
plot_currents_a = current_from_c_rate(NOMINAL_CAPACITY_AH, plot_rates)
plot_durations_h = ideal_duration_hours(plot_rates)

fig, (ax_current, ax_duration) = plt.subplots(1, 2, figsize=(13, 5), sharex=True)

ax_current.plot(plot_rates, plot_currents_a, linewidth=2)
ax_current.scatter(C_RATES, currents_a, s=45, zorder=3)
ax_current.set(
    xscale="log",
    yscale="log",
    xlabel="C-rate",
    ylabel="Current [A]",
    title=f"Current for a {NOMINAL_CAPACITY_AH:.0f} Ah cell",
)

ax_duration.plot(plot_rates, plot_durations_h, linewidth=2)
ax_duration.scatter(C_RATES, durations_h, s=45, zorder=3)
ax_duration.set(
    xscale="log",
    yscale="log",
    xlabel="C-rate",
    ylabel="Ideal duration [h]",
    title="Ideal duration at each C-rate",
)

offsets = [(0, 10), (0, -28), (0, 10), (0, -28), (0, 10), (0, -28), (0, 10)]
for rate, label, current, duration, offset in zip(
    C_RATES, RATE_LABELS, currents_a, durations_h, offsets
):
    ax_current.annotate(
        f"{label}\n{current:.0f} A",
        xy=(rate, current),
        xytext=offset,
        textcoords="offset points",
        ha="center",
        fontsize=8,
    )
    ax_duration.annotate(
        f"{label}\n{pretty_time(float(duration))}",
        xy=(rate, duration),
        xytext=offset,
        textcoords="offset points",
        ha="center",
        fontsize=8,
    )

for ax in (ax_current, ax_duration):
    ax.set_xticks(C_RATES)
    ax.set_xticklabels(RATE_LABELS)
    ax.grid(True, which="both", alpha=0.3)

fig.suptitle("One C-rate table, two complementary views", fontsize=14)
fig.tight_layout()
plt.show()

## 4. Ideal duration is not a real-cell runtime prediction

The relation $t_\mathrm{ideal}=1/C_\mathrm{rate}$ assumes the full nominal capacity is available at the selected current. Real usable capacity depends on the cell and the test conditions.

| Effect | Why it changes measured runtime |
|---|---|
| **Ohmic voltage drop** | A larger current produces a larger instantaneous terminal-voltage drop and can cause the lower cutoff voltage to be reached sooner. |
| **Polarization and transport limitations** | Concentration gradients and reaction overpotentials grow with current, reducing accessible capacity under a fixed cutoff voltage. |
| **Temperature** | Temperature changes resistance, kinetics, transport, and therefore usable capacity. |
| **Chemistry and design** | Electrode thickness, particle size, electrolyte, and other design choices affect rate capability. |
| **State of health** | Aging changes capacity and resistance. |
| **Test definition** | Rated capacity depends on the specified current, temperature, rest conditions, and voltage limits. |

Therefore, 10C corresponds to an **ideal** duration of 6 minutes, but it does not guarantee 6 minutes of real operation. A validated voltage model or measured discharge data is required to predict cutoff time.

This notebook deliberately does not add an empirical rate-capacity curve without cell-specific data.

## 5. Worked example

A datasheet lists a **3.2 Ah** cell with a maximum continuous discharge current of **15 A**. Calculate the corresponding C-rate and ideal duration.

In [ ]:
example_capacity_ah = 3.2
example_current_a = 15.0

example_c_rate = c_rate_from_current(example_current_a, example_capacity_ah)
example_duration_h = ideal_duration_hours(example_c_rate)

print(f"C-rate: {example_c_rate:.2f}C")
print(f"Ideal duration: {example_duration_h:.3f} h")
print(f"Ideal duration: {example_duration_h * 60.0:.1f} min")

The ideal calculation gives approximately **4.69C** and **12.8 minutes**.

That value is a reference calculation, not a guarantee. The actual time before the cell reaches its lower voltage limit must come from a datasheet discharge curve, measured data, or a validated cell model at the relevant temperature and state of health.

## 6. Check yourself

1. A 50 Ah cell is discharged at 25 A. What is its C-rate and ideal duration?
2. Which has the shorter ideal duration at 2C: a 5 Ah cell or a 100 Ah cell?
3. A 20 Ah cell is discharged at 40 A for 25 minutes before reaching its cutoff voltage. How much charge was delivered, and how does that compare with the nominal capacity?
4. Explain why “10C always lasts exactly 6 minutes” is not a valid real-cell prediction.

<details>
<summary><b>Answers</b></summary>

1. $25/50=0.5\mathrm{C}$, so the ideal duration is $1/0.5=2$ hours.
2. Neither. Both have an ideal duration of 30 minutes. Their currents differ: 10 A for the 5 Ah cell and 200 A for the 100 Ah cell.
3. The delivered charge is $40\times(25/60)=16.7$ Ah, which is about 3.3 Ah below the 20 Ah nominal rating under those conditions.
4. Six minutes follows from the ideal reciprocal relation. Real cutoff time also depends on voltage limits, resistance, polarization, temperature, chemistry, design, and state of health.

</details>